In [45]:
# Instal library yang dipakai (lxml = parser HTML, beautifulsoup4 = scraping manual)
!pip -q install lxml beautifulsoup4 kagglehub

import os, re, json, time, hashlib, io, csv
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

# data/raw = data mentah (TIDAK diubah), data/processed = hasil olahan, docs = metadata & bukti
for d in ["data/raw", "data/processed", "docs"]:
    os.makedirs(d, exist_ok=True)

In [46]:
# ==========================================================
# KONFIGURASI IDENTITAS KONTAK UNTUK USER-AGENT
# ==========================================================

import re

# Gunakan email kontak aktif sesuai arahan pengampu.
# Pada versi ini, email pribadi diperbolehkan sesuai arahan terbaru.
EMAIL = "wiraastinarasmawan@gmail.com"
TOPIK = "Destinasi Wisata Indonesia"

if not re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", EMAIL):
    raise ValueError("Masukkan alamat email kontak yang valid.")

USER_AGENT = f"INSTIKI-BIFP243-UTS/1.0 ({EMAIL})"
HEADERS = {"User-Agent": USER_AGENT}

TIMEOUT = 20  # detik
DELAY = 1.5   # detik

print("User-Agent:", USER_AGENT)
print("Email kontak valid:", True)
print("Timeout:", TIMEOUT, "detik")
print("Delay:", DELAY, "detik")


User-Agent: INSTIKI-BIFP243-UTS/1.0 (wiraastinarasmawan@gmail.com)
Email kontak valid: True
Timeout: 20 detik
Delay: 1.5 detik


In [47]:
# ==========================================================
# PROVENANCE DAN HASH MD5
# Memisahkan unduh manual, unduh programatis, akses sumber,
# dan waktu pencatatan metadata.
# ==========================================================

def md5_file(path, chunk=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()


PROVENANCE = []

def catat_provenance(
    nama,
    metode,
    url,
    path,
    lisensi,
    catatan="",
    tanggal_unduh_manual="",
    waktu_akses_sumber_utc="",
    waktu_unduh_programatis_mulai_utc="",
    waktu_unduh_programatis_selesai_utc=""
):
    waktu_pencatatan = datetime.now(
        timezone.utc
    ).strftime("%Y-%m-%d %H:%M:%S UTC")

    PROVENANCE.append({
        "sumber": nama,
        "metode_akuisisi": metode,
        "url": url,
        "tanggal_unduh_manual": tanggal_unduh_manual,
        "waktu_akses_sumber_utc": waktu_akses_sumber_utc,
        "waktu_unduh_programatis_mulai_utc": waktu_unduh_programatis_mulai_utc,
        "waktu_unduh_programatis_selesai_utc": waktu_unduh_programatis_selesai_utc,
        "waktu_pencatatan_metadata_utc": waktu_pencatatan,
        "lisensi": lisensi,
        "berkas_mentah": path,
        "ukuran_byte": os.path.getsize(path),
        "md5": md5_file(path),
        "catatan": catatan
    })

    print(
        f"Provenance tercatat: {nama} | "
        f"MD5={PROVENANCE[-1]['md5']}"
    )


In [48]:
# ==================================================
# KONFIGURASI SUMBER BERKAS DAN BUKTI DATA CARD KAGGLE
# ==================================================

FILE_PATH = "data/raw/tourism_with_id.csv"

KAGGLE_HANDLE = "aprabowo/indonesia-tourism-destination"
FILE_URL = (
    "https://www.kaggle.com/datasets/"
    "aprabowo/indonesia-tourism-destination"
)
URL_KAGGLE_TOS = "https://www.kaggle.com/terms"

# Informasi dari Data Card yang diperiksa:
LISENSI_KAGGLE = (
    "Data files © Original Authors. "
    "Acknowledgements menyatakan 'ALL LICENSE ARE FREE TO USE DATASET', "
    "tetapi lisensi formal pada Data Card tetap tercatat "
    "'Data files © Original Authors'. Izin penggunaan ulang akademik "
    "belum dapat dipastikan hanya dari pernyataan Acknowledgements."
)

ASAL_DATA_KAGGLE = (
    "Data Card Kaggle oleh A_Prabowo menyatakan dataset digunakan dalam "
    "Capstone Project Bangkit Academy 2021 bernama GetLoc dan memberi "
    "kredit kepada GetLoc Team. Repositori proyek GetLoc Apps "
    "(https://github.com/AgungP88/getloc-apps) mencantumkan tim B21-CAP0128. "
    "Namun, sumber awal setiap catatan destinasi dan metode pengumpulan "
    "data tidak dijelaskan secara lengkap pada Data Card."
)

# Jangan ubah menjadi True tanpa bukti dari pemegang hak.
LISENSI_KAGGLE_TERVERIFIKASI = False

# Asal proyek GetLoc diketahui, tetapi sumber asli setiap catatan
# destinasi belum dapat dilacak sepenuhnya.
ASAL_DATA_TERVERIFIKASI = False
ASAL_BUKAN_PLATFORM_TERLARANG = None

# Ubah True setelah kamu sendiri membaca dan memahami Kaggle ToS.
SUDAH_BACA_TOS = False

print("Konfigurasi Kaggle berhasil diisi.")
print("File:", FILE_PATH)
print("URL:", FILE_URL)
print("Handle:", KAGGLE_HANDLE)
print("Lisensi:", LISENSI_KAGGLE)
print("Asal data:", ASAL_DATA_KAGGLE)
print("Lisensi formal terverifikasi:", LISENSI_KAGGLE_TERVERIFIKASI)
print("Asal data terverifikasi:", ASAL_DATA_TERVERIFIKASI)
print("Sudah membaca ToS:", SUDAH_BACA_TOS)


Konfigurasi Kaggle berhasil diisi.
File: data/raw/tourism_with_id.csv
URL: https://www.kaggle.com/datasets/aprabowo/indonesia-tourism-destination
Handle: aprabowo/indonesia-tourism-destination
Lisensi: Data files © Original Authors. Acknowledgements menyatakan 'ALL LICENSE ARE FREE TO USE DATASET', tetapi lisensi formal pada Data Card tetap tercatat 'Data files © Original Authors'. Izin penggunaan ulang akademik belum dapat dipastikan hanya dari pernyataan Acknowledgements.
Asal data: Data Card Kaggle oleh A_Prabowo menyatakan dataset digunakan dalam Capstone Project Bangkit Academy 2021 bernama GetLoc dan memberi kredit kepada GetLoc Team. Repositori proyek GetLoc Apps (https://github.com/AgungP88/getloc-apps) mencantumkan tim B21-CAP0128. Namun, sumber awal setiap catatan destinasi dan metode pengumpulan data tidak dijelaskan secara lengkap pada Data Card.
Lisensi formal terverifikasi: False
Asal data terverifikasi: False
Sudah membaca ToS: False


In [49]:
# ==========================================================
# AKUISISI SATU FILE DATASET KAGGLE MENGGUNAKAN KAGGLEHUB
# ==========================================================

import kagglehub
import shutil
from pathlib import Path

# Hanya unduh file yang benar-benar digunakan oleh proyek.
# KaggleHub mendukung parameter path untuk unduhan satu file.
WAKTU_MULAI_UNDUH_KAGGLE = datetime.now(timezone.utc)

downloaded_file_path = kagglehub.dataset_download(
    KAGGLE_HANDLE,
    path="tourism_with_id.csv"
)

WAKTU_SELESAI_UNDUH_KAGGLE = datetime.now(timezone.utc)

sumber_csv = Path(downloaded_file_path)

if not sumber_csv.is_file():
    raise FileNotFoundError(
        f"File tourism_with_id.csv tidak ditemukan: {downloaded_file_path}"
    )

Path(FILE_PATH).parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(sumber_csv, FILE_PATH)

assert os.path.exists(FILE_PATH), "Dataset belum tersedia di data/raw."

print("Dataset berhasil diunduh melalui KaggleHub.")
print("File hasil unduhan:", downloaded_file_path)
print("File proyek:", FILE_PATH)
print(f"Ukuran: {os.path.getsize(FILE_PATH) / 1024:.2f} KB")
print(
    "Waktu unduh programatis UTC:",
    WAKTU_MULAI_UNDUH_KAGGLE.isoformat(),
    "s.d.",
    WAKTU_SELESAI_UNDUH_KAGGLE.isoformat()
)


Using Colab cache for faster access to the 'indonesia-tourism-destination' dataset.
Dataset berhasil diunduh melalui KaggleHub.
File hasil unduhan: /kaggle/input/indonesia-tourism-destination/tourism_with_id.csv
File proyek: data/raw/tourism_with_id.csv
Ukuran: 357.22 KB
Waktu unduh programatis UTC: 2026-11-19T03:40:48.993355+00:00 s.d. 2026-11-19T03:40:51.280588+00:00


In [50]:
# Deteksi encoding dan delimiter supaya parameter read_csv tepat
raw = open(FILE_PATH, "rb").read(20000)
for enc in ["utf-8", "utf-8-sig", "latin-1"]:
    try:
        sample = raw.decode(enc); ENC = enc; break
    except UnicodeDecodeError:
        continue
try:
    SEP = csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
except Exception:
    SEP = ","
print("encoding:", ENC, "| sep:", repr(SEP))
print(sample[:300])   # intip isi awal

encoding: utf-8 | sep: ','
Place_Id,Place_Name,Description,Category,City,Price,Rating,Time_Minutes,Coordinate,Lat,Long,,
1,Monumen Nasional,"Monumen Nasional atau yang populer disingkat dengan Monas atau Tugu Monas adalah monumen peringatan setinggi 132 meter (433 kaki) yang didirikan untuk mengenang perlawanan dan perjuanga


In [51]:
# Baca berkas dengan parameter yang tepat (sep, encoding, na_values, dtype, skiprows)
df_file = pd.read_csv(
    FILE_PATH,
    sep=SEP,
    encoding=ENC,
    skiprows=0,                                   # tidak ada baris judul/catatan di atas header
    na_values=["-", "--", "NA", "N/A", ""],       # penanda nilai kosong
    dtype={"Place_Id": "int64", "Price": "int64"},  # paksa tipe untuk ID dan harga
)
df_file.head()

,Place_Id,Place_Name,Description,Category,City,Price,Rating,Time_Minutes,Coordinate,Lat,Long,Unnamed: 11,Unnamed: 12
0,1,Monumen Nasional,Monumen Nasional atau yang populer disingkat d...,Budaya,Jakarta,20000,4.6,15.0,"{'lat': -6.1753924, 'lng': 106.8271528}",-6.175392,106.827153,NaN,1
1,2,Kota Tua,"Kota tua di Jakarta, yang juga bernama Kota Tu...",Budaya,Jakarta,0,4.6,90.0,"{'lat': -6.137644799999999, 'lng': 106.8171245}",-6.137645,106.817125,NaN,2
2,3,Dunia Fantasi,Dunia Fantasi atau disebut juga Dufan adalah t...,Taman Hiburan,Jakarta,270000,4.6,360.0,"{'lat': -6.125312399999999, 'lng': 106.8335377}",-6.125312,106.833538,NaN,3
3,4,Taman Mini Indonesia Indah (TMII),Taman Mini Indonesia Indah merupakan suatu kaw...,Taman Hiburan,Jakarta,10000,4.5,NaN,"{'lat': -6.302445899999999, 'lng': 106.8951559}",-6.302446,106.895156,NaN,4
4,5,Atlantis Water Adventure,Atlantis Water Adventure atau dikenal dengan A...,Taman Hiburan,Jakarta,94000,4.5,60.0,"{'lat': -6.12419, 'lng': 106.839134}",-6.124190,106.839134,NaN,5


In [52]:
# Identifikasi dataset: ukuran, tipe data, nilai kosong, cakupan wilayah/waktu
n_baris, n_kolom = df_file.shape
ukuran_byte = os.path.getsize(FILE_PATH)
print(f"Baris: {n_baris} | Kolom: {n_kolom} | Ukuran berkas: {ukuran_byte/1024:.1f} KB")
assert n_baris >= 100, "Dataset harus minimal 100 baris!"

info = pd.DataFrame({
    "tipe_data": df_file.dtypes.astype(str),
    "n_kosong": df_file.isna().sum(),
    "n_unik": df_file.nunique(),
    "contoh_nilai": df_file.iloc[0].astype(str).str.slice(0, 40),
})
display(info)

# Kolom sisa tanpa nama jelas (artefak ekspor); dicatat, tidak diubah di data mentah
kolom_sisa = [c for c in df_file.columns if str(c).startswith("Unnamed")]
print("Kolom tanpa nama (artefak):", kolom_sisa)
print("\nCakupan wilayah (jumlah destinasi per kota):")
print(df_file["City"].value_counts().to_string())
print("\nKategori:", df_file["Category"].value_counts().to_dict())
print("\nRentang nilai numerik:"); print(df_file[["Price", "Rating", "Time_Minutes", "Lat", "Long"]].describe().loc[["min", "max"]].to_string())
print("Kolom bertipe tanggal:", [c for c in df_file.columns if "date" in c.lower() or "tahun" in c.lower()] or "tidak ada")

Baris: 437 | Kolom: 13 | Ukuran berkas: 357.2 KB


,tipe_data,n_kosong,n_unik,contoh_nilai
Place_Id,int64,0,437,1
Place_Name,object,0,437,Monumen Nasional
Description,object,0,437,Monumen Nasional atau yang populer disin
Category,object,0,6,Budaya
City,object,0,5,Jakarta
Price,int64,0,50,20000
Rating,float64,0,14,4.6
Time_Minutes,float64,232,15,15.0
Coordinate,object,0,437,"{'lat': -6.1753924, 'lng': 106.8271528}"
Lat,float64,0,437,-6.1753924


Kolom tanpa nama (artefak): ['Unnamed: 11', 'Unnamed: 12']

Cakupan wilayah (jumlah destinasi per kota):
City
Yogyakarta    126
Bandung       124
Jakarta        84
Semarang       57
Surabaya       46

Kategori: {'Taman Hiburan': 135, 'Budaya': 117, 'Cagar Alam': 106, 'Bahari': 47, 'Tempat Ibadah': 17, 'Pusat Perbelanjaan': 15}

Rentang nilai numerik:
        Price  Rating  Time_Minutes       Lat        Long
min       0.0     3.4          10.0 -8.197894  103.931398
max  900000.0     5.0         360.0  1.078880  112.821662
Kolom bertipe tanggal: tidak ada


In [53]:
# ==========================================================
# PROVENANCE KAGGLE
# ==========================================================

# Tidak mengisi tanggal unduh manual jika tidak ada bukti.
TANGGAL_UNDUH_MANUAL_KAGGLE = ""

catat_provenance(
    nama="Kaggle - Indonesia Tourism Destination",
    metode="Unduh programatis menggunakan KaggleHub",
    url=FILE_URL,
    path=FILE_PATH,
    lisensi=LISENSI_KAGGLE,
    catatan=(
        f"{n_baris} baris x {n_kolom} kolom. "
        "Data Card mengaitkan dataset dengan Capstone Project "
        "Bangkit Academy 2021 (GetLoc) dan memberi kredit kepada GetLoc Team. "
        "Sumber asli sebagian data destinasi masih perlu dikonfirmasi."
    ),
    tanggal_unduh_manual=TANGGAL_UNDUH_MANUAL_KAGGLE,
    waktu_unduh_programatis_mulai_utc=(
        WAKTU_MULAI_UNDUH_KAGGLE.strftime("%Y-%m-%d %H:%M:%S UTC")
    ),
    waktu_unduh_programatis_selesai_utc=(
        WAKTU_SELESAI_UNDUH_KAGGLE.strftime("%Y-%m-%d %H:%M:%S UTC")
    )
)

print(
    "Tanggal unduh manual Kaggle:",
    TANGGAL_UNDUH_MANUAL_KAGGLE or "Tidak dicatat (tidak ada bukti)"
)


Provenance tercatat: Kaggle - Indonesia Tourism Destination | MD5=722ca7f02aa2f847c41798c3f6a5d7a5
Tanggal unduh manual Kaggle: Tidak dicatat (tidak ada bukti)


In [54]:
# Estimasi awal storage 1 tahun untuk CSV Kaggle saja.
# Estimasi seluruh berkas hasil akuisisi dihitung kembali pada cell inventaris Soal 1.
# ASUMSI: berkas diunduh ulang setiap bulan dan disimpan sebagai berkas baru tanpa kompresi;
# cadangan 30% untuk hasil olahan (merge), backup, dan pertumbuhan jumlah destinasi.
ukuran_mb = ukuran_byte / (1024 ** 2)
bulanan = ukuran_mb * 12
cadangan = bulanan * 0.30
total_mb = bulanan + cadangan
print(f"Ukuran berkas      : {ukuran_mb:.4f} MB")
print(f"x 12 bulan         : {bulanan:.4f} MB")
print(f"+ cadangan 30%     : {cadangan:.4f} MB")
print(f"TOTAL estimasi CSV Kaggle/thn : {total_mb:.4f} MB (= {total_mb/1024:.6f} GB)")
ESTIMASI_STORAGE_MB = round(total_mb, 4)

Ukuran berkas      : 0.3489 MB
x 12 bulan         : 4.1862 MB
+ cadangan 30%     : 1.2559 MB
TOTAL estimasi CSV Kaggle/thn : 5.4421 MB (= 0.005315 GB)


In [55]:
# Ringkasan lisensi dan ketentuan tiap sumber (bukti Soal 2)
LISENSI_SUMBER = pd.DataFrame([
    [
        "Kaggle - Indonesia Tourism Destination",
        LISENSI_KAGGLE,
        FILE_URL,
        URL_KAGGLE_TOS
    ],
    [
        "Open-Meteo Historical API",
        "Data API: CC BY 4.0 (atribusi wajib); layanan API publik gratis untuk penggunaan nonkomersial",
        "https://open-meteo.com/en/license",
        "https://open-meteo.com/en/terms"
    ],
    [
        "Wikipedia (en)",
        "CC BY-SA 4.0 (atribusi dan ketentuan share-alike)",
        "https://en.wikipedia.org/wiki/Wikipedia:Text_of_the_Creative_Commons_Attribution-ShareAlike_4.0_International_License",
        "https://foundation.wikimedia.org/wiki/Policy:Terms_of_Use"
    ],
], columns=["sumber", "lisensi_dan_ketentuan", "url_lisensi", "url_ketentuan"])

display(LISENSI_SUMBER)


,sumber,lisensi_dan_ketentuan,url_lisensi,url_ketentuan
0,Kaggle - Indonesia Tourism Destination,Data files © Original Authors. Acknowledgement...,https://www.kaggle.com/datasets/aprabowo/indon...,https://www.kaggle.com/terms
1,Open-Meteo Historical API,Data API: CC BY 4.0 (atribusi wajib); layanan ...,https://open-meteo.com/en/license,https://open-meteo.com/en/terms
2,Wikipedia (en),CC BY-SA 4.0 (atribusi dan ketentuan share-alike),https://en.wikipedia.org/wiki/Wikipedia:Text_o...,https://foundation.wikimedia.org/wiki/Policy:T...


In [56]:
# Deteksi kandidat PII dan quasi-identifier (heuristik nama kolom + pola nilai)
pola_kolom = {
    "PII langsung": r"^(nama|name|email|telp|phone|hp|nik|ktp|alamat|address|npwp|paspor)$|user",
    "Quasi-identifier": r"city|kota|lat|long|coordinate|koordinat|kecamatan|desa|usia|age|gender|kebangsaan",
}
temuan = []
for kol in df_file.columns:
    for jenis, pola in pola_kolom.items():
        if re.search(pola, str(kol), flags=re.I):
            temuan.append({"kolom": kol, "kategori": jenis})

# Pola nilai pada kolom teks: email, telepon Indonesia, NIK 16 digit
pola_nilai = {"email": r"[\w.\-]+@[\w.\-]+\.\w+", "telepon": r"(?:\+62|\b08)\d{8,12}", "nik_16_digit": r"\b\d{16}\b"}
kolom_teks = df_file.select_dtypes(include="object").columns
for kol in kolom_teks:
    for nama, p in pola_nilai.items():
        n = df_file[kol].astype(str).str.contains(p, regex=True).sum()
        if n:
            temuan.append({"kolom": kol, "kategori": f"PII (pola nilai {nama}, {n} baris)"})

pii_df = pd.DataFrame(temuan, columns=["kolom", "kategori"])
display(pii_df)
PII_LANGSUNG = pii_df[pii_df["kategori"].str.startswith("PII")]
QUASI = pii_df[pii_df["kategori"] == "Quasi-identifier"]["kolom"].tolist()
print("PII langsung ditemukan:", len(PII_LANGSUNG), "| quasi-identifier:", QUASI)

,kolom,kategori
0,City,Quasi-identifier
1,Coordinate,Quasi-identifier
2,Lat,Quasi-identifier
3,Long,Quasi-identifier


PII langsung ditemukan: 0 | quasi-identifier: ['City', 'Coordinate', 'Lat', 'Long']


In [57]:

# ==========================================================
# FUNGSI POLITE_GET
# User-Agent, timeout, delay, dan retry
# ==========================================================

import requests
import time

def polite_get(url, params=None, max_retries=3):

    for attempt in range(1, max_retries + 1):

        try:
            response = requests.get(
                url,
                params=params,
                headers=HEADERS,
                timeout=TIMEOUT
            )

        except (
            requests.exceptions.Timeout,
            requests.exceptions.ConnectionError
        ) as e:

            # Retry hanya untuk masalah koneksi
            if attempt == max_retries:
                raise

            waktu_tunggu = DELAY * attempt

            print(
                f"Koneksi gagal (percobaan {attempt}): {e}"
            )
            print(
                f"Mencoba kembali dalam {waktu_tunggu} detik..."
            )

            time.sleep(waktu_tunggu)
            continue

        # --------------------------------------------------
        # HTTP 400-499 selain 429 tidak perlu diulang
        # --------------------------------------------------

        if 400 <= response.status_code < 500:

            if response.status_code != 429:
                print(
                    f"Request ditolak: HTTP {response.status_code}. "
                    "Periksa URL atau parameter."
                )
                response.raise_for_status()

        # --------------------------------------------------
        # HTTP 429 dan 500-599 dapat dicoba kembali
        # --------------------------------------------------

        if (
            response.status_code == 429
            or response.status_code >= 500
        ):

            if attempt == max_retries:
                response.raise_for_status()

            waktu_tunggu = DELAY * attempt

            print(
                f"Server/API error: HTTP {response.status_code}"
            )
            print(
                f"Mencoba kembali dalam {waktu_tunggu} detik..."
            )

            time.sleep(waktu_tunggu)
            continue

        # --------------------------------------------------
        # Jika berhasil
        # --------------------------------------------------

        response.raise_for_status()

        time.sleep(DELAY)

        return response

    raise RuntimeError(
        "Request gagal setelah seluruh percobaan."
    )


In [58]:
for kota, (lat, lon) in LOKASI.items():
    print(kota, lat, lon)
    assert -90 <= lat <= 90
    assert -180 <= lon <= 180

print("Semua koordinat valid. Siap melakukan akuisisi.")

Jakarta -6.2088 106.8456
Yogyakarta -7.7956 110.3695
Bandung -6.9175 107.6191
Semarang -6.9667 110.4167
Surabaya -7.2575 112.7521
Semua koordinat valid. Siap melakukan akuisisi.


In [59]:

# ==========================================================
# AKUISISI DATA CUACA HISTORIS OPEN-METEO - 5 KOTA
# Periode: 1 Januari s.d. 31 Desember 2024
# ==========================================================

import json
import pandas as pd
import os
import hashlib

from pathlib import Path
from datetime import datetime, timezone

# ----------------------------------------------------------
# 1. KONFIGURASI LOKASI
# ----------------------------------------------------------

LOKASI = {
    "Jakarta": (-6.2088, 106.8456),
    "Yogyakarta": (-7.7956, 110.3695),
    "Bandung": (-6.9175, 107.6191),
    "Semarang": (-6.9667, 110.4167),
    "Surabaya": (-7.2575, 112.7521),
}

TGL_MULAI = "2024-01-01"
TGL_AKHIR = "2024-12-31"

URL_API = "https://archive-api.open-meteo.com/v1/archive"

# ----------------------------------------------------------
# 2. VALIDASI KONFIGURASI DAN KOORDINAT
# ----------------------------------------------------------

assert len(LOKASI) == 5, "Harus terdapat 5 kota."

for kota, (lat, lon) in LOKASI.items():

    if not (-90 <= lat <= 90):
        raise ValueError(
            f"Latitude {kota} tidak valid: {lat}"
        )

    if not (-180 <= lon <= 180):
        raise ValueError(
            f"Longitude {kota} tidak valid: {lon}"
        )

if TGL_MULAI > TGL_AKHIR:
    raise ValueError("Tanggal mulai melebihi tanggal akhir.")

Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)

print("Validasi lima kota berhasil.")
print("Periode:", TGL_MULAI, "sampai", TGL_AKHIR)

# ----------------------------------------------------------
# 3. FUNGSI MD5 DAN PROVENANCE
# ----------------------------------------------------------

PROVENANCE = globals().get("PROVENANCE", [])

def md5_file(path):
    h = hashlib.md5()

    with open(path, "rb") as f:
        for blok in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(blok)

    return h.hexdigest()


# Pastikan fungsi provenance menerima waktu akses API.
def catat_provenance(
    nama,
    metode,
    url,
    path,
    lisensi,
    catatan="",
    tanggal_unduh_manual="",
    waktu_akses_sumber_utc=""
):
    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(
            f"Berkas provenance tidak ditemukan: {path}"
        )

    record = {
        "sumber": nama,
        "metode_akuisisi": metode,
        "url": url,
        "tanggal_unduh_manual": tanggal_unduh_manual,
        "waktu_akses_sumber_utc": waktu_akses_sumber_utc,
        "waktu_pencatatan_metadata_utc": (
            datetime.now(timezone.utc).strftime(
                "%Y-%m-%d %H:%M:%S UTC"
            )
        ),
        "lisensi": lisensi,
        "berkas_mentah": str(path),
        "ukuran_byte": path.stat().st_size,
        "md5": md5_file(path),
        "catatan": catatan
    }

    # Perbarui catatan yang sama jika cell dijalankan ulang.
    indeks_lama = next(
        (
            i for i, item in enumerate(PROVENANCE)
            if item.get("sumber") == nama
            and item.get("berkas_mentah") == str(path)
        ),
        None
    )

    if indeks_lama is None:
        PROVENANCE.append(record)
    else:
        PROVENANCE[indeks_lama] = record


# ----------------------------------------------------------
# 4. AKUISISI DATA CUACA LIMA KOTA
# ----------------------------------------------------------

frames = []

for kota, (lat, lon) in LOKASI.items():

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": TGL_MULAI,
        "end_date": TGL_AKHIR,
        "daily": (
            "temperature_2m_mean,"
            "precipitation_sum,"
            "sunshine_duration"
        ),
        "timezone": "Asia/Jakarta"
    }

    print(f"\nMengambil data cuaca: {kota}")

    waktu_mulai_api = datetime.now(timezone.utc)

    # polite_get menggunakan HEADERS, TIMEOUT, DELAY,
    # dan mekanisme retry yang sudah didefinisikan.
    r = polite_get(URL_API, params=params)
    r.raise_for_status()

    waktu_selesai_api = datetime.now(timezone.utc)

    js = r.json()

    if "daily" not in js or "time" not in js["daily"]:
        raise ValueError(
            f"Respons harian {kota} tidak lengkap."
        )

    # ------------------------------------------------------
    # Simpan JSON mentah
    # ------------------------------------------------------

    p_raw = Path(
        f"data/raw/openmeteo_{kota}.json"
    )

    with open(p_raw, "w", encoding="utf-8") as f:
        json.dump(
            js,
            f,
            ensure_ascii=False,
            indent=2
        )

    # ------------------------------------------------------
    # Catat provenance API
    # ------------------------------------------------------

    waktu_akses = (
        waktu_mulai_api.strftime("%Y-%m-%d %H:%M:%S UTC")
        + " sampai "
        + waktu_selesai_api.strftime("%Y-%m-%d %H:%M:%S UTC")
    )

    catat_provenance(
        nama=f"Open-Meteo Historical API - {kota}",
        metode="API (HTTP GET)",
        url=r.url,
        path=str(p_raw),
        lisensi="CC BY 4.0 (atribusi wajib)",
        catatan=(
            f"Data cuaca harian {TGL_MULAI} sampai "
            f"{TGL_AKHIR}; sumber Open-Meteo."
        ),
        waktu_akses_sumber_utc=waktu_akses
    )

    # ------------------------------------------------------
    # Normalisasi JSON menjadi DataFrame
    # ------------------------------------------------------

    d = pd.DataFrame(js["daily"])

    d["lokasi"] = kota
    d["latitude"] = js["latitude"]
    d["longitude"] = js["longitude"]

    if len(d) != 366:
        raise ValueError(
            f"Data {kota} tidak memiliki 366 hari. "
            f"Jumlah ditemukan: {len(d)}"
        )

    frames.append(d)

    print(
        f"Berhasil: {kota}, "
        f"{len(d)} baris data."
    )

# ----------------------------------------------------------
# 5. PENGGABUNGAN DATA LIMA KOTA
# ----------------------------------------------------------

if len(frames) != 5:
    raise ValueError(
        "Tidak semua data lima kota berhasil diakuisisi."
    )

df_api = pd.concat(
    frames,
    ignore_index=True
)

df_api = df_api.rename(
    columns={"time": "tanggal"}
)

df_api["tanggal"] = pd.to_datetime(
    df_api["tanggal"]
)

# ----------------------------------------------------------
# 6. VALIDASI HASIL PENGGABUNGAN
# ----------------------------------------------------------

jumlah_hari = 366
jumlah_kota = len(LOKASI)
jumlah_harapan = jumlah_hari * jumlah_kota

assert len(df_api) == jumlah_harapan, (
    f"Jumlah baris tidak sesuai: {len(df_api)}"
)

assert df_api["lokasi"].nunique() == 5

assert df_api["tanggal"].min() == pd.Timestamp("2024-01-01")

assert df_api["tanggal"].max() == pd.Timestamp("2024-12-31")

assert not df_api.duplicated(
    subset=["tanggal", "lokasi"]
).any(), "Terdapat data tanggal dan kota duplikat."

# ----------------------------------------------------------
# 7. SIMPAN DATA HASIL PENGGABUNGAN
# ----------------------------------------------------------

p_csv = "data/processed/data_api_cuaca.csv"

df_api.to_csv(
    p_csv,
    index=False,
    encoding="utf-8-sig"
)

# ----------------------------------------------------------
# 8. TAMPILKAN HASIL
# ----------------------------------------------------------

print("\nAKUISISI OPEN-METEO BERHASIL")
print("=" * 55)

print("Jumlah kota:", df_api["lokasi"].nunique())
print("Jumlah baris:", len(df_api))
print("Jumlah kolom:", len(df_api.columns))
print(
    "Rentang tanggal:",
    df_api["tanggal"].min(),
    "s.d.",
    df_api["tanggal"].max()
)

print(
    "Ukuran CSV:",
    f"{Path(p_csv).stat().st_size / 1024:.2f} KB"
)

print("\nJumlah data per kota:")
print(df_api.groupby("lokasi").size())

print("\nContoh hasil:")
display(df_api.head())

print("\nData cuaca tersimpan di:", p_csv)


Validasi lima kota berhasil.
Periode: 2024-01-01 sampai 2024-12-31

Mengambil data cuaca: Jakarta
Berhasil: Jakarta, 366 baris data.

Mengambil data cuaca: Yogyakarta
Berhasil: Yogyakarta, 366 baris data.

Mengambil data cuaca: Bandung
Berhasil: Bandung, 366 baris data.

Mengambil data cuaca: Semarang
Berhasil: Semarang, 366 baris data.

Mengambil data cuaca: Surabaya
Berhasil: Surabaya, 366 baris data.

AKUISISI OPEN-METEO BERHASIL
Jumlah kota: 5
Jumlah baris: 1830
Jumlah kolom: 7
Rentang tanggal: 2024-01-01 00:00:00 s.d. 2024-12-31 00:00:00
Ukuran CSV: 105.51 KB

Jumlah data per kota:
lokasi
Bandung       366
Jakarta       366
Semarang      366
Surabaya      366
Yogyakarta    366
dtype: int64

Contoh hasil:


,tanggal,temperature_2m_mean,precipitation_sum,sunshine_duration,lokasi,latitude,longitude
0,2024-01-01,26.8,11.0,26267.57,Jakarta,-6.221441,106.856186
1,2024-01-02,27.4,8.4,27591.26,Jakarta,-6.221441,106.856186
2,2024-01-03,25.8,29.5,8244.02,Jakarta,-6.221441,106.856186
3,2024-01-04,25.2,59.8,4901.31,Jakarta,-6.221441,106.856186
4,2024-01-05,26.6,15.9,39525.22,Jakarta,-6.221441,106.856186



Data cuaca tersimpan di: data/processed/data_api_cuaca.csv


In [60]:
from pathlib import Path

print("BENTUK DATA:", df_api.shape)

print("\nJUMLAH DATA PER KOTA:")
print(df_api.groupby("lokasi").size())

print("\nRENTANG TANGGAL:")
print(df_api["tanggal"].min(), "sampai", df_api["tanggal"].max())

print("\nPEMERIKSAAN FILE JSON:")
for kota in LOKASI:
    file = Path(f"data/raw/openmeteo_{kota}.json")
    print(kota, ":", file.exists())

print("\nFILE CSV:")
print(
    "data_api_cuaca.csv:",
    Path("data/processed/data_api_cuaca.csv").exists()
)

BENTUK DATA: (1830, 7)

JUMLAH DATA PER KOTA:
lokasi
Bandung       366
Jakarta       366
Semarang      366
Surabaya      366
Yogyakarta    366
dtype: int64

RENTANG TANGGAL:
2024-01-01 00:00:00 sampai 2024-12-31 00:00:00

PEMERIKSAAN FILE JSON:
Jakarta : True
Yogyakarta : True
Bandung : True
Semarang : True
Surabaya : True

FILE CSV:
data_api_cuaca.csv: True


In [61]:
# Bukti penanganan status code: permintaan sengaja salah (latitude di luar rentang) -> 4xx
try:
    polite_get(URL_API, params={"latitude": 999, "longitude": 0, "start_date": TGL_MULAI,
                                "end_date": TGL_AKHIR, "daily": "temperature_2m_mean"})
except requests.HTTPError as e:
    print("Status code error ditangani:", e.response.status_code)
except RuntimeError as e:
    print("Gagal setelah retry:", e)

Request ditolak: HTTP 400. Periksa URL atau parameter.
Status code error ditangani: 400


In [62]:
# ✏️ Halaman target dan pencocok tabel
URL_HALAMAN = "https://en.wikipedia.org/wiki/List_of_Indonesian_cities_by_population"

def cek_kepatuhan(url, user_agent=USER_AGENT):
    # robots.txt selalu berada di root domain, bukan di belakang path halaman
    p = urlparse(url)
    robots_url = f"{p.scheme}://{p.netloc}/robots.txt"
    r = polite_get(robots_url)                     # diambil dengan User-Agent kita
    rp = RobotFileParser()
    rp.parse(r.text.splitlines())
    return {
        "halaman": url, "robots_url": robots_url, "status_code_robots": r.status_code,
        "user_agent": user_agent,
        "boleh_diakses": rp.can_fetch(user_agent, url),
        "crawl_delay_ua_kita": rp.crawl_delay(user_agent),
        "waktu_cek_utc": datetime.now(timezone.utc).isoformat(),
    }, r.text

hasil_robots, isi_robots = cek_kepatuhan(URL_HALAMAN)
Path("data/raw/robots_txt_evidence.txt").write_text(isi_robots, encoding="utf-8")
Path("docs/robots_check.json").write_text(json.dumps(hasil_robots, indent=2), encoding="utf-8")
print(json.dumps(hasil_robots, indent=2))

# Hentikan jika tidak diizinkan; jika ada crawl-delay untuk UA kita, pakai sebagai jeda
assert hasil_robots["boleh_diakses"], "Dilarang scraping: robots.txt tidak mengizinkan halaman ini"
if hasil_robots["crawl_delay_ua_kita"]:
    DELAY = max(DELAY, hasil_robots["crawl_delay_ua_kita"])
catat_provenance(
    "Wikipedia robots.txt",
    "HTTP GET",
    hasil_robots["robots_url"],
    "data/raw/robots_txt_evidence.txt",
    "n/a (bukti pemeriksaan kepatuhan)",
    catatan="Bukti pemeriksaan robots.txt untuk halaman Wikipedia yang dituju.",
    waktu_akses_sumber_utc=hasil_robots["waktu_cek_utc"]
)

{
  "halaman": "https://en.wikipedia.org/wiki/List_of_Indonesian_cities_by_population",
  "robots_url": "https://en.wikipedia.org/robots.txt",
  "status_code_robots": 200,
  "user_agent": "INSTIKI-BIFP243-UTS/1.0 (wiraastinarasmawan@gmail.com)",
  "boleh_diakses": true,
  "crawl_delay_ua_kita": null,
  "waktu_cek_utc": "2026-11-19T03:41:03.923943+00:00"
}


In [63]:
# Ambil halaman (timeout, UA, jeda, cek status ada di polite_get) dan simpan HTML mentah
waktu_akses_wikipedia_mulai = datetime.now(timezone.utc)
r = polite_get(URL_HALAMAN)
waktu_akses_wikipedia_selesai = datetime.now(timezone.utc)

Path("data/raw/halaman_wikipedia.html").write_text(r.text, encoding="utf-8")
catat_provenance(
    "Wikipedia - List of Indonesian cities by population",
    "web scraping statis",
    URL_HALAMAN,
    "data/raw/halaman_wikipedia.html",
    "CC BY-SA 4.0",
    "HTML mentah; atribusi Wikipedia",
    waktu_akses_sumber_utc=(
        waktu_akses_wikipedia_mulai.strftime("%Y-%m-%d %H:%M:%S UTC")
        + " s.d. "
        + waktu_akses_wikipedia_selesai.strftime("%Y-%m-%d %H:%M:%S UTC")
    )
)

soup = BeautifulSoup(r.text, "lxml")                # lxml: cepat dan toleran
print("Judul halaman :", soup.title.string.strip())
print("Jumlah <table>:", len(soup.select("table")))

Judul halaman : List of Indonesian cities by population - Wikipedia
Jumlah <table>: 9


In [64]:
# Ekstraksi tabel: cara cepat (pandas.read_html) vs cara manual (BeautifulSoup + selector)
# Tabel dipilih lewat nama kolom ("City" dan "Province"), bukan nomor urut (lebih tahan perubahan)
kandidat = pd.read_html(io.StringIO(r.text), match="Province")
df_cepat = next(t for t in kandidat if {"City", "Province"} <= set(map(str, t.columns)))

tabel_manual = None
for t in soup.select("table.wikitable"):
    header = [th.get_text(strip=True) for th in t.select("tr")[0].select("th")]
    if "City" in header and "Province" in header:
        tabel_manual = t; break
assert tabel_manual is not None, "Tabel target tidak ditemukan"
rows = [[re.sub(r"\[.*?\]", "", c.get_text(" ", strip=True)) for c in tr.select("th, td")]
        for tr in tabel_manual.select("tr")[1:]]
df_manual = pd.DataFrame(rows)

print("read_html:", df_cepat.shape, "| manual:", df_manual.shape)
assert df_cepat.shape[0] == df_manual.shape[0], "Jumlah baris cara cepat dan manual berbeda"

# Bersihkan penanda catatan kaki seperti [3] dari teks dan rapikan nama kolom
df_scrape = df_cepat.copy()
for kol in ["City", "Province", "Region"]:
    df_scrape[kol] = df_scrape[kol].astype(str).str.replace(r"\[.*?\]", "", regex=True).str.strip()
df_scrape = df_scrape.rename(columns={"2020 census [2]": "Population_2020"})[["City", "Province", "Region", "Population_2020"]]
df_scrape.to_csv("data/processed/tabel_wikipedia_kota.csv", index=False)
display(df_scrape.head(10))

read_html: (95, 8) | manual: (95, 8)


,City,Province,Region,Population_2020
0,Jakarta,Special Capital Region of Jakarta,Java,10562088
1,Surabaya,East Java,Java,2874314
2,Bekasi,West Java,Java,2543676
3,Bandung,West Java,Java,2444160
4,Medan,North Sumatra,Sumatra,2435252
5,Depok,West Java,Java,2056335
6,Tangerang,Banten,Java,1895486
7,Palembang,South Sumatra,Sumatra,1668848
8,Semarang,Central Java,Java,1653524
9,Makassar,South Sulawesi,Sulawesi,1423877


In [65]:
# Uji apakah konten ada di HTML awal (jika tidak, halaman dirender JavaScript dan perlu Selenium/Playwright)
ada_di_html = all(k in r.text for k in ["Surabaya", "Yogyakarta"])
print("Konten ada di HTML awal:", ada_di_html,
      "->", "requests + BeautifulSoup cukup" if ada_di_html else "kemungkinan dirender JavaScript")

# Metadata scraping (verifikasi ulang hasil)
meta_scraping = {
    "berkas_html": "data/raw/halaman_wikipedia.html",
    "berkas_tabel": "data/processed/tabel_wikipedia_kota.csv",
    "situs_sumber": URL_HALAMAN,
    "waktu_scraping_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "user_agent": USER_AGENT,
    "jeda_antar_request_detik": DELAY,
    "timeout_detik": TIMEOUT,
    "jumlah_record": int(len(df_scrape)),
    "robots_txt_mengizinkan": hasil_robots["boleh_diakses"],
    "bukti_robots": "data/raw/robots_txt_evidence.txt",
    "lisensi_konten": "CC BY-SA 4.0 (Wikipedia), atribusi wajib",
}
Path("docs/scraping_meta.json").write_text(json.dumps(meta_scraping, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(meta_scraping, indent=2, ensure_ascii=False))

Konten ada di HTML awal: True -> requests + BeautifulSoup cukup
{
  "berkas_html": "data/raw/halaman_wikipedia.html",
  "berkas_tabel": "data/processed/tabel_wikipedia_kota.csv",
  "situs_sumber": "https://en.wikipedia.org/wiki/List_of_Indonesian_cities_by_population",
  "waktu_scraping_utc": "2026-11-19T03:41:05+00:00",
  "user_agent": "INSTIKI-BIFP243-UTS/1.0 (wiraastinarasmawan@gmail.com)",
  "jeda_antar_request_detik": 1.5,
  "timeout_detik": 20,
  "jumlah_record": 95,
  "robots_txt_mengizinkan": true,
  "bukti_robots": "data/raw/robots_txt_evidence.txt",
  "lisensi_konten": "CC BY-SA 4.0 (Wikipedia), atribusi wajib"
}


In [66]:
# ==========================================================
# SOAL 1: INVENTARIS JENIS, FORMAT, TIPE, CAKUPAN, DAN STORAGE
# ==========================================================

from pathlib import Path
import os
import pandas as pd

folder_raw = Path("data/raw")
if not folder_raw.exists():
    raise FileNotFoundError("Folder data/raw belum tersedia.")

inventaris = []

for path in sorted(folder_raw.glob("*")):
    if not path.is_file():
        continue

    ext = path.suffix.lower()
    jenis = {
        ".csv": "Data tabular",
        ".json": "Data terstruktur (JSON)",
        ".html": "Halaman web (HTML)",
        ".txt": "Dokumentasi/bukti teks",
        ".pdf": "Dokumen pendukung (PDF)"
    }.get(ext, "Format lain")

    ukuran = path.stat().st_size

    inventaris.append({
        "nama_berkas": path.name,
        "jenis_data": jenis,
        "format": ext or "tanpa ekstensi",
        "ukuran_byte": ukuran,
        "ukuran_KB": round(ukuran / 1024, 2),
        "ukuran_MB": round(ukuran / (1024 ** 2), 5),
    })

if not inventaris:
    raise RuntimeError("Belum ada berkas di data/raw untuk diinventarisasi.")

inventaris_df = pd.DataFrame(inventaris)
print("INVENTARIS BERKAS MENTAH DAN PENDUKUNG")
display(inventaris_df)

# PDF instruksi tugas bukan data hasil akuisisi, sehingga tidak dimasukkan
# dalam proyeksi penyimpanan dataset sumber.
berkas_data = inventaris_df[
    inventaris_df["format"].isin([".csv", ".json", ".html", ".txt"])
].copy()

total_raw_byte = int(berkas_data["ukuran_byte"].sum())
total_raw_mb = total_raw_byte / (1024 ** 2)

print(f"Jumlah seluruh berkas di data/raw: {len(inventaris_df)}")
print(f"Jumlah berkas data/akuisisi: {len(berkas_data)}")
print(f"Total ukuran data hasil akuisisi: {total_raw_mb:.5f} MB")

# Karakteristik dataset destinasi wisata.
print("\nKARAKTERISTIK DATASET DESTINASI WISATA")
print("Jumlah baris:", len(df_file))
print("Jumlah kolom:", len(df_file.columns))
print("Format file: CSV (data tabular)")
print("Sumber:", FILE_URL)

display(pd.DataFrame({
    "nama_kolom": df_file.columns,
    "tipe_data": [str(df_file[c].dtype) for c in df_file.columns],
    "jumlah_kosong": [int(df_file[c].isna().sum()) for c in df_file.columns],
}).reset_index(drop=True))

print("\nCAKUPAN KOTA DESTINASI")
display(
    df_file["City"].value_counts()
    .rename_axis("kota")
    .reset_index(name="jumlah_destinasi")
)

print("\nCAKUPAN DATA CUACA")
print("Jumlah baris cuaca:", len(df_api))
print("Jumlah kota:", df_api["lokasi"].nunique())
print("Kota:", sorted(df_api["lokasi"].dropna().unique().tolist()))
print("Rentang tanggal:", df_api["tanggal"].min(), "s.d.", df_api["tanggal"].max())
print("Kolom dan tipe data cuaca:")
display(df_api.dtypes.rename("tipe_data").to_frame())

# Estimasi storage tahunan:
# Asumsi satu paket data hasil akuisisi disimpan setiap bulan,
# ditambah cadangan 30% untuk variasi ukuran dan metadata.
frekuensi_arsip_per_tahun = 12
cadangan_persen = 30
estimasi_dasar_mb = total_raw_mb * frekuensi_arsip_per_tahun
estimasi_cadangan_mb = estimasi_dasar_mb * cadangan_persen / 100
estimasi_storage_mb = estimasi_dasar_mb + estimasi_cadangan_mb

ESTIMASI_STORAGE_MB = estimasi_storage_mb

print("\nESTIMASI STORAGE SATU TAHUN")
print(f"Ukuran satu paket data      : {total_raw_mb:.5f} MB")
print(f"Frekuensi penyimpanan       : {frekuensi_arsip_per_tahun} kali/tahun")
print(f"Ukuran dasar tahunan        : {estimasi_dasar_mb:.5f} MB")
print(f"Cadangan {cadangan_persen}%              : {estimasi_cadangan_mb:.5f} MB")
print(f"Estimasi total              : {estimasi_storage_mb:.5f} MB")
print(
    "Asumsi: satu snapshot data saat ini diarsipkan setiap bulan "
    "dengan ukuran tetap dan cadangan 30%. Ini proyeksi, bukan ukuran "
    "aktual yang dijamin."
)


INVENTARIS BERKAS MENTAH DAN PENDUKUNG


,nama_berkas,jenis_data,format,ukuran_byte,ukuran_KB,ukuran_MB
0,halaman_wikipedia.html,Halaman web (HTML),.html,287608,280.87,0.27428
1,openmeteo_Bandung.json,Data terstruktur (JSON),.json,22091,21.57,0.02107
2,openmeteo_Jakarta.json,Data terstruktur (JSON),.json,22117,21.60,0.02109
3,openmeteo_Semarang.json,Data terstruktur (JSON),.json,22059,21.54,0.02104
4,openmeteo_Surabaya.json,Data terstruktur (JSON),.json,22060,21.54,0.02104
5,openmeteo_Yogyakarta.json,Data terstruktur (JSON),.json,22092,21.57,0.02107
6,robots_txt_evidence.txt,Dokumentasi/bukti teks,.txt,28275,27.61,0.02697
7,tourism_with_id.csv,Data tabular,.csv,365796,357.22,0.34885


Jumlah seluruh berkas di data/raw: 8
Jumlah berkas data/akuisisi: 8
Total ukuran data hasil akuisisi: 0.75540 MB

KARAKTERISTIK DATASET DESTINASI WISATA
Jumlah baris: 437
Jumlah kolom: 13
Format file: CSV (data tabular)
Sumber: https://www.kaggle.com/datasets/aprabowo/indonesia-tourism-destination


,nama_kolom,tipe_data,jumlah_kosong
0,Place_Id,int64,0
1,Place_Name,object,0
2,Description,object,0
3,Category,object,0
4,City,object,0
5,Price,int64,0
6,Rating,float64,0
7,Time_Minutes,float64,232
8,Coordinate,object,0
9,Lat,float64,0



CAKUPAN KOTA DESTINASI


,kota,jumlah_destinasi
0,Yogyakarta,126
1,Bandung,124
2,Jakarta,84
3,Semarang,57
4,Surabaya,46



CAKUPAN DATA CUACA
Jumlah baris cuaca: 1830
Jumlah kota: 5
Kota: ['Bandung', 'Jakarta', 'Semarang', 'Surabaya', 'Yogyakarta']
Rentang tanggal: 2024-01-01 00:00:00 s.d. 2024-12-31 00:00:00
Kolom dan tipe data cuaca:


,tipe_data
tanggal,datetime64[ns]
temperature_2m_mean,float64
precipitation_sum,float64
sunshine_duration,float64
lokasi,object
latitude,float64
longitude,float64



ESTIMASI STORAGE SATU TAHUN
Ukuran satu paket data      : 0.75540 MB
Frekuensi penyimpanan       : 12 kali/tahun
Ukuran dasar tahunan        : 9.06484 MB
Cadangan 30%              : 2.71945 MB
Estimasi total              : 11.78429 MB
Asumsi: satu snapshot data saat ini diarsipkan setiap bulan dengan ukuran tetap dan cadangan 30%. Ini proyeksi, bukan ukuran aktual yang dijamin.


In [67]:
# Salinan kerja; data mentah tidak diubah. Buang dua kolom artefak ekspor.
df_a = df_file.drop(columns=kolom_sisa).copy()
df_a["City"] = df_a["City"].str.strip()

# Agregasi cuaca harian -> satu baris per kota (kunci kanan unik)
cuaca_kota = (df_api.groupby("lokasi", as_index=False)
              .agg(suhu_rata2_c=("temperature_2m_mean", "mean"),
                   curah_hujan_total_mm=("precipitation_sum", "sum"),
                   durasi_sinar_matahari_rata2_detik=("sunshine_duration", "mean"))
              .rename(columns={"lokasi": "City"}))
wilayah = df_scrape.drop_duplicates("City")

print("Kunci unik (cuaca):", cuaca_kota["City"].is_unique, "| (wilayah):", wilayah["City"].is_unique)
print("Kota di dataset tanpa pasangan di cuaca :", set(df_a["City"]) - set(cuaca_kota["City"]))
print("Kota di dataset tanpa pasangan di wilayah:", set(df_a["City"]) - set(wilayah["City"]))

Kunci unik (cuaca): True | (wilayah): True
Kota di dataset tanpa pasangan di cuaca : set()
Kota di dataset tanpa pasangan di wilayah: set()


In [68]:
# Merge bertahap dengan validate="many_to_one": kiri boleh berulang, kanan harus unik (jika tidak -> MergeError)
n_awal = len(df_a)
merged = (df_a
          .merge(cuaca_kota, on="City", how="left", validate="many_to_one", indicator="_m_cuaca")
          .merge(wilayah, on="City", how="left", validate="many_to_one", indicator="_m_wilayah"))

# Bukti tidak ada duplikasi baris dan semua baris punya pasangan
assert len(merged) == n_awal, "Jumlah baris berubah -> ada duplikasi!"
assert not merged["Place_Id"].duplicated().any(), "Place_Id duplikat"
assert (merged["_m_cuaca"] == "both").all() and (merged["_m_wilayah"] == "both").all(), "Ada baris tanpa pasangan"
print(f"Baris awal: {n_awal} | hasil merge: {len(merged)}")
print("Duplikat Place_Id:", int(merged["Place_Id"].duplicated().sum()), "| duplikat baris penuh:", int(merged.drop(columns=['_m_cuaca','_m_wilayah']).duplicated().sum()))
merged = merged.drop(columns=["_m_cuaca", "_m_wilayah"])
merged.head()

Baris awal: 437 | hasil merge: 437
Duplikat Place_Id: 0 | duplikat baris penuh: 0


,Place_Id,Place_Name,Description,Category,City,Price,Rating,Time_Minutes,Coordinate,Lat,Long,suhu_rata2_c,curah_hujan_total_mm,durasi_sinar_matahari_rata2_detik,Province,Region,Population_2020
0,1,Monumen Nasional,Monumen Nasional atau yang populer disingkat d...,Budaya,Jakarta,20000,4.6,15.0,"{'lat': -6.1753924, 'lng': 106.8271528}",-6.175392,106.827153,27.397268,2500.2,35343.827322,Special Capital Region of Jakarta,Java,10562088
1,2,Kota Tua,"Kota tua di Jakarta, yang juga bernama Kota Tu...",Budaya,Jakarta,0,4.6,90.0,"{'lat': -6.137644799999999, 'lng': 106.8171245}",-6.137645,106.817125,27.397268,2500.2,35343.827322,Special Capital Region of Jakarta,Java,10562088
2,3,Dunia Fantasi,Dunia Fantasi atau disebut juga Dufan adalah t...,Taman Hiburan,Jakarta,270000,4.6,360.0,"{'lat': -6.125312399999999, 'lng': 106.8335377}",-6.125312,106.833538,27.397268,2500.2,35343.827322,Special Capital Region of Jakarta,Java,10562088
3,4,Taman Mini Indonesia Indah (TMII),Taman Mini Indonesia Indah merupakan suatu kaw...,Taman Hiburan,Jakarta,10000,4.5,NaN,"{'lat': -6.302445899999999, 'lng': 106.8951559}",-6.302446,106.895156,27.397268,2500.2,35343.827322,Special Capital Region of Jakarta,Java,10562088
4,5,Atlantis Water Adventure,Atlantis Water Adventure atau dikenal dengan A...,Taman Hiburan,Jakarta,94000,4.5,60.0,"{'lat': -6.12419, 'lng': 106.839134}",-6.124190,106.839134,27.397268,2500.2,35343.827322,Special Capital Region of Jakarta,Java,10562088


In [69]:
# Ringkasan dan bukti hasil penggabungan
print("PRESENTASI HASIL MERGE DATA")
print("=" * 55)
print("Jumlah baris sebelum merge :", len(df_a))
print("Jumlah baris sesudah merge :", len(merged))
print("Jumlah kolom hasil         :", len(merged.columns))
print("Place_Id unik              :", merged["Place_Id"].nunique())
print("Duplikasi Place_Id         :", int(merged["Place_Id"].duplicated().sum()))
print("Nilai kosong per kolom:")
display(merged.isna().sum().rename("jumlah_kosong").to_frame())
print("Contoh hasil penggabungan:")
display(merged.head(10))
print("Sumber data yang digabung:")
print("- Dataset destinasi wisata Kaggle")
print("- Data cuaca harian Open-Meteo untuk lima kota")
print("- Data wilayah hasil ekstraksi tabel Wikipedia")


PRESENTASI HASIL MERGE DATA
Jumlah baris sebelum merge : 437
Jumlah baris sesudah merge : 437
Jumlah kolom hasil         : 17
Place_Id unik              : 437
Duplikasi Place_Id         : 0
Nilai kosong per kolom:


,jumlah_kosong
Place_Id,0
Place_Name,0
Description,0
Category,0
City,0
Price,0
Rating,0
Time_Minutes,232
Coordinate,0
Lat,0


Contoh hasil penggabungan:


,Place_Id,Place_Name,Description,Category,City,Price,Rating,Time_Minutes,Coordinate,Lat,Long,suhu_rata2_c,curah_hujan_total_mm,durasi_sinar_matahari_rata2_detik,Province,Region,Population_2020
0,1,Monumen Nasional,Monumen Nasional atau yang populer disingkat d...,Budaya,Jakarta,20000,4.6,15.0,"{'lat': -6.1753924, 'lng': 106.8271528}",-6.175392,106.827153,27.397268,2500.2,35343.827322,Special Capital Region of Jakarta,Java,10562088
1,2,Kota Tua,"Kota tua di Jakarta, yang juga bernama Kota Tu...",Budaya,Jakarta,0,4.6,90.0,"{'lat': -6.137644799999999, 'lng': 106.8171245}",-6.137645,106.817125,27.397268,2500.2,35343.827322,Special Capital Region of Jakarta,Java,10562088
2,3,Dunia Fantasi,Dunia Fantasi atau disebut juga Dufan adalah t...,Taman Hiburan,Jakarta,270000,4.6,360.0,"{'lat': -6.125312399999999, 'lng': 106.8335377}",-6.125312,106.833538,27.397268,2500.2,35343.827322,Special Capital Region of Jakarta,Java,10562088
3,4,Taman Mini Indonesia Indah (TMII),Taman Mini Indonesia Indah merupakan suatu kaw...,Taman Hiburan,Jakarta,10000,4.5,NaN,"{'lat': -6.302445899999999, 'lng': 106.8951559}",-6.302446,106.895156,27.397268,2500.2,35343.827322,Special Capital Region of Jakarta,Java,10562088
4,5,Atlantis Water Adventure,Atlantis Water Adventure atau dikenal dengan A...,Taman Hiburan,Jakarta,94000,4.5,60.0,"{'lat': -6.12419, 'lng': 106.839134}",-6.124190,106.839134,27.397268,2500.2,35343.827322,Special Capital Region of Jakarta,Java,10562088
5,6,Taman Impian Jaya Ancol,Taman Impian Jaya Ancol merupakan sebuah objek...,Taman Hiburan,Jakarta,25000,4.5,10.0,"{'lat': -6.117333200000001, 'lng': 106.8579951}",-6.117333,106.857995,27.397268,2500.2,35343.827322,Special Capital Region of Jakarta,Java,10562088
6,7,Kebun Binatang Ragunan,Kebun Binatang Ragunan adalah sebuah kebun bin...,Cagar Alam,Jakarta,4000,4.5,NaN,"{'lat': -6.3124593, 'lng': 106.8201865}",-6.312459,106.820187,27.397268,2500.2,35343.827322,Special Capital Region of Jakarta,Java,10562088
7,8,Ocean Ecopark,Ocean Ecopark Salah satu zona rekreasi Ancol y...,Taman Hiburan,Jakarta,180000,4.0,NaN,"{'lat': -6.125801699999999, 'lng': 106.8363249}",-6.125802,106.836325,27.397268,2500.2,35343.827322,Special Capital Region of Jakarta,Java,10562088
8,9,Pelabuhan Marina,Pelabuhan Marina Ancol berada di kawasan Taman...,Bahari,Jakarta,175000,4.4,NaN,"{'lat': 1.07888, 'lng': 103.931398}",1.078880,103.931398,27.397268,2500.2,35343.827322,Special Capital Region of Jakarta,Java,10562088
9,10,Pulau Tidung,Pulau Tidung adalah salah satu kelurahan di ke...,Bahari,Jakarta,150000,4.5,NaN,"{'lat': -5.803205300000001, 'lng': 106.5237907}",-5.803205,106.523791,27.397268,2500.2,35343.827322,Special Capital Region of Jakarta,Java,10562088


Sumber data yang digabung:
- Dataset destinasi wisata Kaggle
- Data cuaca harian Open-Meteo untuk lima kota
- Data wilayah hasil ekstraksi tabel Wikipedia


In [70]:
# Simpan hasil gabungan beserta metadata
OUT = "data/processed/merged_dataset.csv"
merged.to_csv(OUT, index=False)
metadata = {
    "judul": TOPIK,
    "dibuat_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "email_kontak": EMAIL,
    "sumber": [p["sumber"] for p in PROVENANCE],
    "jumlah_baris_berkas": int(n_awal),
    "jumlah_baris_api": int(len(df_api)),
    "jumlah_baris_scraping": int(len(df_scrape)),
    "jumlah_baris_hasil": int(len(merged)),
    "jumlah_kolom_hasil": int(merged.shape[1]),
    "kolom": list(merged.columns),
    "kunci_merge": ["City"],
    "validate": "many_to_one",
    "metode_merge": "left",
    "duplikasi_place_id": int(merged["Place_Id"].duplicated().sum()),
    "kolom_dibuang": kolom_sisa,
    "satuan": {"Price": "rupiah", "Time_Minutes": "menit", "suhu_rata2_c": "derajat Celsius",
               "curah_hujan_total_mm": "mm per tahun 2024", "durasi_sinar_matahari_rata2_detik": "detik per hari"},
    "file_hasil": OUT,
    "md5_hasil": md5_file(OUT),
}
Path("data/processed/merged_dataset.metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(metadata, indent=2, ensure_ascii=False)[:1800])

{
  "judul": "Destinasi Wisata Indonesia",
  "dibuat_utc": "2026-11-19T03:41:06+00:00",
  "email_kontak": "wiraastinarasmawan@gmail.com",
  "sumber": [
    "Kaggle - Indonesia Tourism Destination",
    "Open-Meteo Historical API - Jakarta",
    "Open-Meteo Historical API - Yogyakarta",
    "Open-Meteo Historical API - Bandung",
    "Open-Meteo Historical API - Semarang",
    "Open-Meteo Historical API - Surabaya",
    "Wikipedia robots.txt",
    "Wikipedia - List of Indonesian cities by population"
  ],
  "jumlah_baris_berkas": 437,
  "jumlah_baris_api": 1830,
  "jumlah_baris_scraping": 95,
  "jumlah_baris_hasil": 437,
  "jumlah_kolom_hasil": 17,
  "kolom": [
    "Place_Id",
    "Place_Name",
    "Description",
    "Category",
    "City",
    "Price",
    "Rating",
    "Time_Minutes",
    "Coordinate",
    "Lat",
    "Long",
    "suhu_rata2_c",
    "curah_hujan_total_mm",
    "durasi_sinar_matahari_rata2_detik",
    "Province",
    "Region",
    "Population_2020"
  ],
  "kunci_merge": 

In [71]:
# ==========================================================
# DATA ETHICS CHECKLIST - 9 BUTIR
# ==========================================================

import pandas as pd
import re
from pathlib import Path

# ----------------------------------------------------------
# 1. STATUS VERIFIKASI LISENSI DAN ASAL DATA
# ----------------------------------------------------------

# True hanya jika lisensi formal dan izin penggunaan akademik
# sudah didukung bukti dari pemilik/pemegang hak.
LISENSI_KAGGLE_TERVERIFIKASI = globals().get(
    "LISENSI_KAGGLE_TERVERIFIKASI", False
)

# True setelah benar-benar membaca Kaggle Terms of Use.
SUDAH_BACA_TOS = globals().get(
    "SUDAH_BACA_TOS", False
)

# True setelah sumber asli dataset sudah dikonfirmasi.
ASAL_DATA_TERVERIFIKASI = globals().get(
    "ASAL_DATA_TERVERIFIKASI", False
)

ASAL_BUKAN_PLATFORM_TERLARANG = globals().get(
    "ASAL_BUKAN_PLATFORM_TERLARANG", None
)

# ----------------------------------------------------------
# 2. PEMERIKSAAN EMAIL PRIBADI DAN USER-AGENT
# ----------------------------------------------------------

EMAIL = str(globals().get("EMAIL", "")).strip()
HEADERS = globals().get("HEADERS", {})
USER_AGENT = str(HEADERS.get("User-Agent", ""))

TIMEOUT = globals().get("TIMEOUT", 0)
DELAY = globals().get("DELAY", 0)

# Email pribadi/Gmail digunakan sesuai arahan terbaru yang disampaikan.
email_kontak_ok = bool(
    re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", EMAIL)
)

user_agent_ok = (
    email_kontak_ok
    and EMAIL.lower() in USER_AGENT.lower()
)

timeout_ok = TIMEOUT > 0
delay_ok = DELAY > 0

status_user_agent = (
    "Ya"
    if email_kontak_ok and user_agent_ok and timeout_ok and delay_ok
    else "Perlu perbaikan"
)

# ----------------------------------------------------------
# 3. STATUS LISENSI DAN ASAL DATA
# ----------------------------------------------------------

LISENSI_KAGGLE = globals().get("LISENSI_KAGGLE", "Belum dicatat")
ASAL_DATA_KAGGLE = globals().get("ASAL_DATA_KAGGLE", "Belum dijelaskan")
FILE_URL = globals().get("FILE_URL", "")
URL_KAGGLE_TOS = globals().get(
    "URL_KAGGLE_TOS",
    "https://www.kaggle.com/terms"
)

lisensi_ok = LISENSI_KAGGLE_TERVERIFIKASI is True

asal_ok = (
    ASAL_DATA_TERVERIFIKASI is True
    and bool(ASAL_DATA_KAGGLE.strip())
    and ASAL_BUKAN_PLATFORM_TERLARANG is True
)

status_asal = (
    "Ya"
    if ASAL_BUKAN_PLATFORM_TERLARANG is True
    else "Tidak"
    if ASAL_BUKAN_PLATFORM_TERLARANG is False
    else "Belum dipastikan"
)

status_lisensi = "Ya" if lisensi_ok else "Perlu verifikasi"

status_tos_asal = (
    "Ya"
    if SUDAH_BACA_TOS is True and asal_ok
    else "Perlu verifikasi"
)

# ----------------------------------------------------------
# 4. PEMERIKSAAN PII DAN QUASI-IDENTIFIER
# ----------------------------------------------------------

PII_LANGSUNG = globals().get("PII_LANGSUNG", [])
QUASI = globals().get("QUASI", [])

status_pii = (
    "Ya" if len(PII_LANGSUNG) == 0
    else "Perlu ditinjau"
)

# ----------------------------------------------------------
# 5. PEMERIKSAAN ROBOTS.TXT
# ----------------------------------------------------------

hasil_robots = globals().get("hasil_robots", {})

if hasil_robots:
    status_robots = (
        "Ya" if hasil_robots.get("boleh_diakses") is True
        else "Tidak"
    )
    bukti_robots = (
        f"URL: {hasil_robots.get('robots_url', 'Tidak tersedia')}; "
        f"status HTTP: {hasil_robots.get('status_code_robots', 'Tidak tersedia')}; "
        f"can_fetch={hasil_robots.get('boleh_diakses')}"
    )
else:
    status_robots = "Perlu verifikasi"
    bukti_robots = "Hasil pemeriksaan robots.txt belum tersedia."

# ----------------------------------------------------------
# 6. PEMERIKSAAN PROVENANCE
# ----------------------------------------------------------

PROVENANCE = globals().get("PROVENANCE", [])

provenance_lengkap = (
    len(PROVENANCE) >= 8
    and all(p.get("md5") and os.path.exists(p.get("berkas_mentah", ""))
            for p in PROVENANCE)
)

status_provenance = (
    "Ya" if provenance_lengkap
    else "Perlu dilengkapi"
)

# ----------------------------------------------------------
# 7. MEMBUAT CHECKLIST 9 BUTIR
# ----------------------------------------------------------

checklist = pd.DataFrame([
    [
        1,
        "Tujuan pengumpulan jelas dan relevan",
        "Ya",
        "UTS Data Acquisition & Management; topik destinasi wisata Indonesia dan cuaca."
    ],
    [
        2,
        "Lisensi dataset jelas dan mengizinkan penggunaan akademik",
        status_lisensi,
        f"Data Card mencantumkan: {LISENSI_KAGGLE}. "
        f"Lisensi formal terverifikasi: {lisensi_ok}. "
        f"Data Card: {FILE_URL}. "
        "Pernyataan Acknowledgements tidak menggantikan verifikasi izin formal."
    ],
    [
        3,
        "Ketentuan portal sumber dibaca dan dipatuhi; asal data bukan platform terlarang",
        status_tos_asal,
        f"ToS: {URL_KAGGLE_TOS}. "
        f"Sudah membaca ToS: {SUDAH_BACA_TOS}. "
        f"Asal data: {ASAL_DATA_KAGGLE}. "
        f"Asal terverifikasi: {ASAL_DATA_TERVERIFIKASI}. "
        f"Bukan platform terlarang: {status_asal}."
    ],
    [
        4,
        "Tidak ditemukan PII langsung berdasarkan pemeriksaan",
        status_pii,
        f"Pemindaian menemukan {len(PII_LANGSUNG)} kandidat PII langsung. "
        "Hasil pemindaian otomatis tetap perlu ditinjau."
    ],
    [
        5,
        "Risiko quasi-identifier dan re-identifikasi dinilai",
        "Ya",
        f"Quasi-identifier: {', '.join(QUASI)}. "
        "Risiko re-identifikasi dinilai berdasarkan karakteristik data lokasi."
    ],
    [
        6,
        "robots.txt diperiksa dan halaman yang di-scrape diizinkan",
        status_robots,
        bukti_robots
    ],
    [
        7,
        "Timeout, User-Agent berisi email kontak, dan jeda antar-request digunakan",
        status_user_agent,
        f"User-Agent: {USER_AGENT}; "
        f"email kontak valid: {email_kontak_ok}; "
        f"email tercantum dalam User-Agent: {user_agent_ok}; "
        f"timeout: {TIMEOUT} detik; jeda: {DELAY} detik; "
        "retry menggunakan backoff melalui polite_get."
    ],
    [
        8,
        "API key/token tidak ditulis secara langsung di notebook",
        "Ya",
        "Open-Meteo tidak memerlukan API key untuk endpoint yang digunakan. "
        "Jika Kaggle API digunakan, kredensial disimpan melalui Colab Secrets."
    ],
    [
        9,
        "Atribusi sumber dan provenance dicatat",
        status_provenance,
        f"Jumlah sumber: {len(PROVENANCE)}. "
        "Tanggal unduh manual Kaggle tidak diklaim tanpa bukti; "
        "waktu unduh programatis dan pencatatan metadata dibedakan."
    ],
], columns=["No", "Butir Etika", "Status", "Bukti"])

# ----------------------------------------------------------
# 8. SIMPAN HASIL CHECKLIST
# ----------------------------------------------------------

Path("data/processed").mkdir(parents=True, exist_ok=True)

checklist.to_csv(
    "data/processed/data_ethics_checklist.csv",
    index=False,
    encoding="utf-8-sig"
)

# ----------------------------------------------------------
# 9. TAMPILKAN HASIL
# ----------------------------------------------------------

print("DATA ETHICS CHECKLIST - 9 BUTIR")
print("=" * 70)
display(checklist)

print("\nRINGKASAN STATUS")
print(checklist["Status"].value_counts().to_string())

print("\nPEMERIKSAAN USER-AGENT")
print("Email kontak valid:", email_kontak_ok)
print("Email ada di User-Agent:", user_agent_ok)
print("Timeout:", TIMEOUT)
print("Delay:", DELAY)
print("Status nomor 7:", status_user_agent)

print("\nFile berhasil disimpan:")
print("data/processed/data_ethics_checklist.csv")


DATA ETHICS CHECKLIST - 9 BUTIR


,No,Butir Etika,Status,Bukti
0,1,Tujuan pengumpulan jelas dan relevan,Ya,UTS Data Acquisition & Management; topik desti...
1,2,Lisensi dataset jelas dan mengizinkan pengguna...,Perlu verifikasi,Data Card mencantumkan: Data files © Original ...
2,3,Ketentuan portal sumber dibaca dan dipatuhi; a...,Perlu verifikasi,ToS: https://www.kaggle.com/terms. Sudah memba...
3,4,Tidak ditemukan PII langsung berdasarkan pemer...,Ya,Pemindaian menemukan 0 kandidat PII langsung. ...
4,5,Risiko quasi-identifier dan re-identifikasi di...,Ya,"Quasi-identifier: City, Coordinate, Lat, Long...."
5,6,robots.txt diperiksa dan halaman yang di-scrap...,Ya,URL: https://en.wikipedia.org/robots.txt; stat...
6,7,"Timeout, User-Agent berisi email kontak, dan j...",Ya,User-Agent: INSTIKI-BIFP243-UTS/1.0 (wiraastin...
7,8,API key/token tidak ditulis secara langsung di...,Ya,Open-Meteo tidak memerlukan API key untuk endp...
8,9,Atribusi sumber dan provenance dicatat,Ya,Jumlah sumber: 8. Tanggal unduh manual Kaggle ...



RINGKASAN STATUS
Status
Ya                  7
Perlu verifikasi    2

PEMERIKSAAN USER-AGENT
Email kontak valid: True
Email ada di User-Agent: True
Timeout: 20
Delay: 1.5
Status nomor 7: Ya

File berhasil disimpan:
data/processed/data_ethics_checklist.csv


In [72]:
# ==========================================================
# SIMPAN DAN PERIKSA PROVENANCE
# ==========================================================

prov = pd.DataFrame(PROVENANCE)

Path("docs").mkdir(parents=True, exist_ok=True)

prov.to_csv(
    "docs/provenance.csv",
    index=False,
    encoding="utf-8-sig"
)

kolom_tampil = [
    "sumber",
    "metode_akuisisi",
    "tanggal_unduh_manual",
    "waktu_akses_sumber_utc",
    "waktu_unduh_programatis_mulai_utc",
    "waktu_unduh_programatis_selesai_utc",
    "waktu_pencatatan_metadata_utc",
    "lisensi",
    "md5"
]

display(prov[kolom_tampil])

# Verifikasi keberadaan file dan kecocokan MD5.
for item in PROVENANCE:
    path = item["berkas_mentah"]
    assert os.path.exists(path), f"Berkas tidak ditemukan: {path}"
    assert md5_file(path) == item["md5"], (
        f"MD5 tidak cocok: {path}"
    )

print("\nJumlah sumber dalam provenance:", len(PROVENANCE))
print("Semua berkas mentah tersedia dan MD5 cocok.")
print("Provenance berhasil disimpan di docs/provenance.csv.")


,sumber,metode_akuisisi,tanggal_unduh_manual,waktu_akses_sumber_utc,waktu_unduh_programatis_mulai_utc,waktu_unduh_programatis_selesai_utc,waktu_pencatatan_metadata_utc,lisensi,md5
0,Kaggle - Indonesia Tourism Destination,Unduh programatis menggunakan KaggleHub,,,2026-11-19 03:40:48 UTC,2026-11-19 03:40:51 UTC,2026-11-19 03:40:51 UTC,Data files © Original Authors. Acknowledgement...,722ca7f02aa2f847c41798c3f6a5d7a5
1,Open-Meteo Historical API - Jakarta,API (HTTP GET),,2026-11-19 03:40:51 UTC sampai 2026-11-19 03:4...,NaN,NaN,2026-11-19 03:40:53 UTC,CC BY 4.0 (atribusi wajib),418b6cbdc615e2475b42ebe2245074e7
2,Open-Meteo Historical API - Yogyakarta,API (HTTP GET),,2026-11-19 03:40:53 UTC sampai 2026-11-19 03:4...,NaN,NaN,2026-11-19 03:40:55 UTC,CC BY 4.0 (atribusi wajib),4a77d16816fbb0c0911f53e8cf7857af
3,Open-Meteo Historical API - Bandung,API (HTTP GET),,2026-11-19 03:40:55 UTC sampai 2026-11-19 03:4...,NaN,NaN,2026-11-19 03:40:57 UTC,CC BY 4.0 (atribusi wajib),c8cffa636e224b429538f23b80a039a7
4,Open-Meteo Historical API - Semarang,API (HTTP GET),,2026-11-19 03:40:57 UTC sampai 2026-11-19 03:4...,NaN,NaN,2026-11-19 03:40:59 UTC,CC BY 4.0 (atribusi wajib),a507994ee04fa4a87fff2ef81c555209
5,Open-Meteo Historical API - Surabaya,API (HTTP GET),,2026-11-19 03:40:59 UTC sampai 2026-11-19 03:4...,NaN,NaN,2026-11-19 03:41:01 UTC,CC BY 4.0 (atribusi wajib),99941e5dd4ac208c64f9a8a86be4c72e
6,Wikipedia robots.txt,HTTP GET,,2026-11-19T03:41:03.923943+00:00,NaN,NaN,2026-11-19 03:41:03 UTC,n/a (bukti pemeriksaan kepatuhan),fc15d80fd6a4afd9f22975e740c1b48c
7,Wikipedia - List of Indonesian cities by popul...,web scraping statis,,2026-11-19 03:41:03 UTC s.d. 2026-11-19 03:41:...,NaN,NaN,2026-11-19 03:41:05 UTC,CC BY-SA 4.0,48b8c56097ff29de184c2b9abbe9e249



Jumlah sumber dalam provenance: 8
Semua berkas mentah tersedia dan MD5 cocok.
Provenance berhasil disimpan di docs/provenance.csv.


In [73]:
# ==========================================================
# PEMERIKSAAN AKHIR KETENTUAN UTS
# ==========================================================

cek = [
    ("Dataset destinasi memiliki minimal 100 baris (Soal 1)", n_baris >= 100),
    ("Karakteristik data dan tipe kolom sudah ditampilkan (Soal 1)", len(df_file.columns) > 0),
    ("Estimasi storage tahunan sudah dihitung (Soal 1)", ESTIMASI_STORAGE_MB > 0),
    ("Lisensi Kaggle terverifikasi (Soal 2)", LISENSI_KAGGLE_TERVERIFIKASI is True),
    (
        "ToS dibaca dan asal dataset terkonfirmasi (Soal 2)",
        SUDAH_BACA_TOS is True
        and ASAL_DATA_TERVERIFIKASI is True
        and ASAL_BUKAN_PLATFORM_TERLARANG is True
    ),
    ("API Open-Meteo menghasilkan 1.830 baris cuaca (Soal 3)", len(df_api) == 1830),
    ("robots.txt mengizinkan halaman scraping (Soal 4)", hasil_robots["boleh_diakses"] is True),
    ("Tabel hasil scraping tersimpan (Soal 4)", os.path.exists("data/processed/tabel_wikipedia_kota.csv")),
    (
        "Hasil merge tidak menambah baris dan Place_Id tetap unik (Soal 5)",
        len(merged) == n_awal and not merged["Place_Id"].duplicated().any()
    ),
    ("User-Agent berisi email kontak valid dengan timeout dan delay (butir 7)", status_user_agent == "Ya"),
    (
        "Minimal delapan sumber dicatat dengan MD5 dan berkas tersedia",
        len(PROVENANCE) >= 8
        and all(
            p.get("md5")
            and os.path.exists(p.get("berkas_mentah", ""))
            for p in PROVENANCE
        )
    ),
]

print("PEMERIKSAAN AKHIR")
print("=" * 65)
for nama, ok in cek:
    print(("LULUS: " if ok else "PERLU TINDAKAN: ") + nama)

print(
    "\nCatatan: status lisensi dan asal data hanya lulus jika bukti "
    "formal telah diperiksa. Status tidak boleh diubah hanya untuk "
    "mengubah tampilan checklist."
)


PEMERIKSAAN AKHIR
LULUS: Dataset destinasi memiliki minimal 100 baris (Soal 1)
LULUS: Karakteristik data dan tipe kolom sudah ditampilkan (Soal 1)
LULUS: Estimasi storage tahunan sudah dihitung (Soal 1)
PERLU TINDAKAN: Lisensi Kaggle terverifikasi (Soal 2)
PERLU TINDAKAN: ToS dibaca dan asal dataset terkonfirmasi (Soal 2)
LULUS: API Open-Meteo menghasilkan 1.830 baris cuaca (Soal 3)
LULUS: robots.txt mengizinkan halaman scraping (Soal 4)
LULUS: Tabel hasil scraping tersimpan (Soal 4)
LULUS: Hasil merge tidak menambah baris dan Place_Id tetap unik (Soal 5)
LULUS: User-Agent berisi email kontak valid dengan timeout dan delay (butir 7)
LULUS: Minimal delapan sumber dicatat dengan MD5 dan berkas tersedia

Catatan: status lisensi dan asal data hanya lulus jika bukti formal telah diperiksa. Status tidak boleh diubah hanya untuk mengubah tampilan checklist.


In [74]:
# ==========================================================
# README DAN PAKET PENGUMPULAN
# ==========================================================

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

readme = """# UTS Data Acquisition and Management – Indonesia Tourism

## Deskripsi
Proyek ini melakukan akuisisi, pemeriksaan, pengolahan, dan penggabungan data destinasi wisata Indonesia dengan data cuaca historis dan informasi wilayah pendukung.

## Sumber Data
- Kaggle – Indonesia Tourism Destination: https://www.kaggle.com/datasets/aprabowo/indonesia-tourism-destination
- Open-Meteo Historical API: https://open-meteo.com/
- Wikipedia – List of Indonesian cities by population: https://en.wikipedia.org/wiki/List_of_Indonesian_cities_by_population

## Cakupan
- Dataset destinasi: sekitar 400 destinasi di Jakarta, Yogyakarta, Semarang, Bandung, dan Surabaya.
- Cuaca historis: 1 Januari–31 Desember 2024 untuk lima kota tersebut.
- Data pendukung wilayah: tabel kota dari Wikipedia.

## Struktur Repository
- `UTSDAM_perbaikan_final.ipynb`: satu notebook utama untuk seluruh tahapan soal.
- `data/raw/`: sumber mentah CSV, JSON, HTML, dan bukti robots.txt.
- `data/processed/`: hasil olahan, dataset gabungan, metadata, dan checklist etika.
- `docs/`: provenance, bukti robots.txt, dan metadata scraping.

## Tahapan Pengerjaan
1. Akuisisi dataset destinasi wisata dari Kaggle menggunakan KaggleHub.
2. Identifikasi jenis, format, tipe kolom, cakupan, ukuran file, dan estimasi storage tahunan.
3. Akuisisi cuaca harian Open-Meteo untuk lima kota selama tahun 2024.
4. Pemeriksaan robots.txt dan ekstraksi tabel kota dari Wikipedia.
5. Pembersihan data dan penggabungan berdasarkan kolom `City`.
6. Validasi jumlah baris, keunikan `Place_Id`, dan kelengkapan hasil merge.
7. Pemeriksaan etika dan pencatatan provenance setiap sumber.

## Catatan Lisensi dan Etika
- Data Card Kaggle menampilkan `Data files © Original Authors`; pernyataan pada bagian Acknowledgements tidak dengan sendirinya membuktikan izin lisensi formal untuk penggunaan ulang.
- Status lisensi dan asal asli dataset Kaggle ditandai perlu verifikasi sampai tersedia bukti yang memadai.
- Data API Open-Meteo dilisensikan CC BY 4.0 dan memerlukan atribusi. Penggunaan layanan API publik gratis dibatasi untuk penggunaan nonkomersial; periksa ketentuan resmi sebelum penggunaan lain.
- User-Agent menggunakan email kontak yang diisi pada konfigurasi notebook. Pastikan alamat dan kebijakan penggunaannya sesuai arahan dosen.
- Tanggal unduh manual Kaggle tidak diisi tanpa bukti. Waktu unduh programatis melalui KaggleHub dicatat secara terpisah dari waktu pencatatan metadata.

## Cara Menjalankan
1. Buka notebook `UTSDAM_perbaikan_final.ipynb` di Google Colab.
2. Jalankan semua cell dari atas ke bawah tanpa melewati cell.
3. Pastikan dataset KaggleHub berhasil diunduh dan semua request sumber data berhasil.
4. Periksa keluaran setiap cell dan file yang terbentuk pada folder `data/` dan `docs/`.

## Hasil Utama
Hasil penggabungan disimpan pada `data/processed/merged_dataset.csv`, dengan informasi struktur dan validasi di `data/processed/merged_dataset.metadata.json`.
"""

Path("README.md").write_text(readme, encoding="utf-8")
print("README.md berhasil dibuat.")

# Kemas folder data, dokumentasi, dan README.
# Notebook akan dimasukkan jika file .ipynb tersedia di runtime.
arsip = "UTS_DAM_Indonesia_Tourism.zip"

file_paket = []
for folder in ["data", "docs"]:
    base = Path(folder)
    if base.exists():
        file_paket.extend([x for x in base.rglob("*") if x.is_file()])

file_paket.append(Path("README.md"))

# Jika notebook telah diunggah ke runtime dengan nama final, masukkan ke ZIP.
notebook_runtime = Path("UTSDAM_perbaikan_final.ipynb")
if notebook_runtime.exists():
    file_paket.append(notebook_runtime)
    print("Notebook .ipynb akan dimasukkan ke dalam ZIP.")
else:
    print(
        "PERHATIAN: file notebook .ipynb tidak ditemukan di runtime. "
        "Unduh notebook dari File > Download > Download .ipynb dan "
        "unggah bersama ZIP ke ELSA/GitHub."
    )

with ZipFile(arsip, "w", compression=ZIP_DEFLATED) as zf:
    for f in file_paket:
        zf.write(f, arcname=str(f))

print("Arsip berhasil dibuat:", arsip)
print("Jumlah file di dalam arsip:", len(file_paket))

try:
    from google.colab import files
    files.download(arsip)
except ImportError:
    print("Di luar Google Colab; arsip tersedia pada lokasi kerja.")


README.md berhasil dibuat.
PERHATIAN: file notebook .ipynb tidak ditemukan di runtime. Unduh notebook dari File > Download > Download .ipynb dan unggah bersama ZIP ke ELSA/GitHub.
Arsip berhasil dibuat: UTS_DAM_Indonesia_Tourism.zip
Jumlah file di dalam arsip: 17


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>